In [2]:
!pip install --upgrade datasets

In [3]:
import torch
import transformers
import sagemaker
from sagemaker.huggingface import HuggingFace
from torch.utils.data import DataLoader, Dataset, random_split
from sagemaker import get_execution_role
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from transformers import DistilBertTokenizer,DistilBertModel
from datasets import load_dataset
import boto3

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


2026-03-02 13:55:57.962905: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
role = get_execution_role()

In [6]:
sess = sagemaker.Session()
# sagemaker session bucket -> used for uploading data, models and logs
# sagemaker will automatically create this bucket if it not exists
sagemaker_session_bucket=None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()


In [7]:
tokenizer_name = 'distilbert-base-uncased'

In [6]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

In [12]:
class CSVDataset(Dataset):
    def __init__(self, path,field_names=None):
        # 1. Load data
        if field_names is not None:
            df = pd.read_csv(path,sep='\t',names= field_names)
        else:
            df = pd.read_csv(path,sep='\t')

        df = df[['TITLE','CATEGORY']]

        my_dict = {'b':'Business',
           'e':'Entertainment',
           'm':'Health',
           't': 'Technology'}

        df['CATEGORY'] = df['CATEGORY'].apply(lambda x: my_dict[x])
        
        # 2. Encode the label field
        self.encoder = LabelEncoder()
        df['label'] = self.encoder.fit_transform(df['CATEGORY'])
        
        # 3. Separate features and labels
        # Assuming features are all columns except 'label'
        self.X = df['TITLE'].values
        self.y = df['label'].values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {'text':self.X[idx], label=self.y[idx])

In [14]:
s3_path = 's3://sagemaker-eu-north-1-910119346557/training-data/newsCorpora.csv'

In [15]:
field_names= ['ID','TITLE','URL','PUBLISHER','CATEGORY','STORY','HOSTNAME','TIMESTAMP']

In [16]:
df = pd.read_csv(s3_path,sep='\t',names= field_names)

In [17]:
df = df[['TITLE','CATEGORY']]

my_dict = {'b':'Business',
           'e':'Entertainment',
           'm':'Health',
           't': 'Technology'
}

df['CATEGORY'] = df['CATEGORY'].apply(lambda x: my_dict[x])

encode_dict = {}

def encode_cat(x):
    if x not in encode_dict.keys():
        encode_dict[x] = len(encode_dict)
    return encode_dict[x]

df['ENCODE_CAT'] = df['CATEGORY'].apply(lambda x: encode_cat(x))

In [21]:
df.rename(columns={'TITLE': 'text', 'ENCODE_CAT':'label'}).drop(columns='CATEGORY').to_csv('data/newscorpora.csv',index=False)

In [20]:
training_full = load_dataset('csv', data_files='data/newscorpora.csv')

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 training_full = load_dataset('csv', data_files='data/newscorpora.csv',split=['train', 't     │
│   2                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/datasets/load.py:1517 in load_dataset                    │
│                                                                                                  │
│   1514 │   keep_in_memory = (                                                                    │
│   1515 │   │   keep_in_memory if keep_in_memory is not None else is_small_dataset(builder_insta  │
│   1516 │   )                                                                                     │
│ ❱ 1517 │   ds = builder_instance.as_dataset(split=split, verification_mode=verification_mode, i  │
│   1518 │                                                                                         │
│   1519 │   return ds                                                                             │
│   1520                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/datasets/builder.py:1052 in as_dataset                   │
│                                                                                                  │
│   1049 │   │   verification_mode = VerificationMode(verification_mode or VerificationMode.BASIC  │
│   1050 │   │                                                                                     │
│   1051 │   │   # Create a dataset for each of the given splits                                   │
│ ❱ 1052 │   │   datasets = map_nested(                                                            │
│   1053 │   │   │   partial(                                                                      │
│   1054 │   │   │   │   self._build_single_dataset,                                               │
│   1055 │   │   │   │   run_post_process=run_post_process,                                        │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/datasets/utils/py_utils.py:521 in map_nested             │
│                                                                                                  │
│   518 │   │   │   │   batch_size = max(len(iterable) // num_proc + int(len(iterable) % num_pro   │
│   519 │   │   │   iterable = list(iter_batched(iterable, batch_size))                            │
│   520 │   │   mapped = [                                                                         │
│ ❱ 521 │   │   │   _single_map_nested((function, obj, batched, batch_size, types, None, True, N   │
│   522 │   │   │   for obj in hf_tqdm(iterable, disable=disable_tqdm, desc=desc)                  │
│   523 │   │   ]                                                                                  │
│   524 │   │   if batched:                                                                        │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/datasets/utils/py_utils.py:382 in _single_map_nested     │
│                                                                                                  │
│   379 │   │   if batched:                                                                        │
│   380 │   │   │   return function([data_struct])[0]        

In [26]:
training_full = training_full['train']

In [42]:
training_test = training_full.train_test_split(train_size=0.8)

In [43]:
training_test

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 337935
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 84484
    })
})

In [49]:
# tokenizer helper function

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', max_length=20,truncation=True)

In [50]:
# tokenize dataset
train_dataset = training_test['train'].map(tokenize, batched=True)
test_dataset = training_test['test'].map(tokenize, batched=True)

Map:   0%|          | 0/337935 [00:00<?, ? examples/s]

Map:   0%|          | 0/84484 [00:00<?, ? examples/s]

In [53]:
# set format for pytorch
train_dataset =  train_dataset.rename_column("label", "labels")
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

In [8]:
s3_prefix = 'datasets/newscorpora'

# save train_dataset to s3
training_input_path = f's3://{sess.default_bucket()}/{s3_prefix}/train'
#train_dataset.save_to_disk(training_input_path)

# save test_dataset to s3
test_input_path = f's3://{sess.default_bucket()}/{s3_prefix}/test'
#test_dataset.save_to_disk(test_input_path)

In [9]:
# hyperparameters, which are passed into the training job
hyperparameters={'epochs': 1,
                 'train_batch_size': 32,
                 'model_name':'distilbert-base-uncased'
                 }

In [12]:
huggingface_estimator = HuggingFace(entry_point='training.py',
                            source_dir='./scripts',
                            instance_type='ml.p3.2xlarge',
                            instance_count=1,
                            role=role,
                            transformers_version='4.26',
                            pytorch_version='1.13',
                            py_version='py39',
                            hyperparameters = hyperparameters)

In [ ]:
huggingface_estimator = HuggingFace(
    entry_point='training.py',
    source_dir='scripts/',
    role=role,
    instance_count=1, 
    instance_type='ml.g4dn.xlarge',
    transformers_version='4.6',
    pytorch_version= '1.8',
    output_path='s3://sagemaker-eu-north-1-910119346557/output/', 
    py_version= 'py39',
    hyperparameters = hyperparameters,
    enable_sagemaker_metrics = True
)

In [13]:
# starting the train job with our uploaded datasets as input
huggingface_estimator.fit({'train': training_input_path, 'test': test_input_path})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-03-02-13-57-00-174
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # starting the train job with our uploaded datasets as input                                 │
│ ❱ 2 huggingface_estimator.fit({'train': training_input_path, 'test': test_input_path})           │
│   3                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:167 in wrapper  │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:138 in wrapper  │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:1373 in fit                       │
│                                                                                                  │
│   1370 │   │   self._prepare_for_training(job_name=job_name

In [66]:
from datasets import load_from_disk

In [67]:
training_input_path

's3://sagemaker-eu-north-1-910119346557/datasets/newscorpora/train'

In [72]:
train_dataset = load_from_disk(training_input_path)

In [73]:
train_dataset

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 337935
})

In [74]:
len(train_dataset)

337935